# LIGO GW150914: an hour of gravitational-wave strain

GWOSC publishes 4096-second strain files at 4096 Hz and 16384 Hz.
That is 16.8 million or 67.1 million samples in a single line. The first
chart keeps the complete raw record so XY can exercise M4 decimation. The
event detail then extracts 320 ms around `t = 0` and applies a tapered
35–350 Hz FFT bandpass so the chirp is legible without presenting the
result as a whitened or template-matched detection product.

The default 4 kHz HDF5 file is about 134 MB. Set
`LIGO_SAMPLE_RATE=16384` for the full-rate, roughly 536 MB file.

**Source:** [GWOSC GW150914 event page](https://gwosc.org/events/GW150914/),
the [official Hanford template reconstruction](https://gwosc.org/GW150914data/P150914/fig2-unfiltered-template-reconstruction-H.txt),
and [GWOSC URL lookup documentation](https://gwosc.readthedocs.io/en/stable/locate.html).
GWOSC event data are released under CC BY 4.0; follow the
acknowledgement guidance linked from the event page.

Install beside XY with
`python -m pip install numpy requests h5py gwosc xy`.


In [ ]:
import os
from pathlib import Path

import h5py
import numpy as np
import requests
from gwosc.locate import get_event_urls

import xy

DATA_DIR = Path(os.getenv("XY_REAL_WORLD_DATA", "data")) / "gwosc"
DATA_DIR.mkdir(parents=True, exist_ok=True)

EVENT = "GW150914"
EVENT_GPS = 1_126_259_462.4
DETECTOR = os.getenv("LIGO_DETECTOR", "H1")
SAMPLE_RATE = int(os.getenv("LIGO_SAMPLE_RATE", "4096"))
DURATION = 4096
DETECTOR_SITES = {"H1": "HANFORD", "L1": "LIVINGSTON"}
if SAMPLE_RATE not in {4096, 16384}:
    raise ValueError("LIGO_SAMPLE_RATE must be 4096 or 16384")
if DETECTOR not in DETECTOR_SITES:
    raise ValueError("LIGO_DETECTOR must be H1 or L1")

urls = get_event_urls(
    EVENT,
    catalog="GWTC-1-confident",
    version=3,
    detector=DETECTOR,
    duration=DURATION,
    sample_rate=SAMPLE_RATE,
    format="hdf5",
)
if not urls:
    raise RuntimeError("GWOSC returned no matching strain file")
url = urls[0]
hdf5_path = DATA_DIR / Path(url).name
if not hdf5_path.exists():
    with requests.get(
        url,
        stream=True,
        timeout=(30, 3600),
    ) as response:
        response.raise_for_status()
        partial = hdf5_path.with_suffix(".hdf5.part")
        with partial.open("wb") as output:
            for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):
                output.write(chunk)
        partial.replace(hdf5_path)

reconstruction_url = (
    "https://gwosc.org/GW150914data/P150914/fig2-unfiltered-template-reconstruction-H.txt"
)
reconstruction_path = DATA_DIR / Path(reconstruction_url).name
if not reconstruction_path.exists():
    with requests.get(reconstruction_url, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        partial = reconstruction_path.with_suffix(".txt.part")
        with partial.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                output.write(chunk)
        partial.replace(reconstruction_path)

print(f"cached strain file: {hdf5_path}")
print(f"cached reconstruction: {reconstruction_path}")

In [ ]:
with h5py.File(hdf5_path, "r") as data:
    strain = np.asarray(data["strain"]["Strain"], dtype=np.float64)
    gps_start = float(np.asarray(data["meta"]["GPSstart"]))
    x_spacing = float(
        data["strain"]["Strain"].attrs.get(
            "Xspacing",
            1 / SAMPLE_RATE,
        )
    )

seconds_from_event = gps_start + np.arange(strain.size, dtype=np.float64) * x_spacing - EVENT_GPS
print(
    f"{strain.size:,} samples · {1 / x_spacing:,.0f} Hz · "
    f"{strain.nbytes / 2**20:,.1f} MiB canonical strain"
)

In [ ]:
full_x_bounds = (float(seconds_from_event[0]), float(seconds_from_event[-1]))
full_y_bounds = (float(np.nanmin(strain)), float(np.nanmax(strain)))

SIGNAL_FONT = "IBM Plex Mono, ui-monospace, monospace"
SIGNAL_AXIS_STYLE = {
    "grid_color": "#173640",
    "grid_width": 1,
    "grid_dash": "dotted",
    "grid_opacity": 0.62,
    "axis_color": "#3d7380",
    "axis_width": 1,
    "tick_color": "#43f4ff",
    "tick_width": 1,
    "tick_length": 5,
    "tick_label_color": "#a9ccd2",
    "label_color": "#d7f9fc",
    "tick_label_size": 12,
    "label_size": 12,
}
SIGNAL_THEME = xy.theme(
    background="#03070b",
    plot_background="#061116",
    text_color="#d7f9fc",
    grid_color="#173640",
    axis_color="#3d7380",
    crosshair_color="#ff4f91",
    selection_color="#43f4ff",
    selection_fill="#43f4ff22",
)
SIGNAL_STYLES = {
    "tick_label": {"font_family": SIGNAL_FONT},
    "axis_title": {"font_family": SIGNAL_FONT, "letter_spacing": "0.05em"},
    "annotation_label": {"font_family": SIGNAL_FONT},
    "tooltip": {
        "background": "#07151a",
        "color": "#d7f9fc",
        "border": "1px solid #2b7680",
        "border_radius": 5,
        "font_family": SIGNAL_FONT,
    },
}
SIGNAL_CARD = {"border": "1px solid #173640", "font_family": SIGNAL_FONT}

overview_chart = xy.line_chart(
    xy.line(
        seconds_from_event,
        strain,
        name=f"{DETECTOR} · raw strain",
        color="#43f4ff",
        width=1.0,
        opacity=0.72,
    ),
    xy.vline(0, color="#ff4f91", width=2, opacity=0.95, style={"dash": "6,5"}),
    xy.text(
        0.111,
        0.925,
        "4096-SECOND RAW RECORD",
        dx=0,
        dy=0,
        color="#f1fdff",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 20,
            "font_weight": 700,
            "letter_spacing": "0.04em",
        },
    ),
    xy.text(
        0.111,
        0.85,
        f"{EVENT}  /  {DETECTOR}  /  {1 / x_spacing:,.0f} HZ  /  {strain.size / 1e6:.1f}M SAMPLES",
        dx=0,
        dy=0,
        color="#79a7af",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 10,
            "font_weight": 700,
            "letter_spacing": "0.09em",
        },
    ),
    xy.x_axis(
        label="TIME FROM EVENT  /  seconds",
        domain=full_x_bounds,
        bounds=full_x_bounds,
        style=SIGNAL_AXIS_STYLE,
    ),
    xy.y_axis(
        label="RAW DETECTOR STRAIN  h(t)",
        label_offset=-16,
        bounds=full_y_bounds,
        style=SIGNAL_AXIS_STYLE,
    ),
    xy.tooltip(title=f"{DETECTOR} RAW STRAIN", format={"x": "+.1f", "y": ".3e"}),
    xy.legend(show=False),
    xy.interaction_config(crosshair=True, wheel_zoom=True, box_zoom=True),
    SIGNAL_THEME,
    styles=SIGNAL_STYLES,
    style=SIGNAL_CARD,
    width=1150,
    height=360,
    padding=(82, 32, 66, 128),
)
overview_payload = overview_chart.figure().build_payload()[0]
print("overview render tier:", overview_payload["traces"][0]["tier"])
overview_chart

## Event detail: a truthful signal-first view

A Hann-tapered segment with a linearly rolled frequency response suppresses
frequencies below 25 Hz, passes 35–350 Hz, and rolls off by 400 Hz. The
chart labels that processing directly and keeps the GWOSC event reference
at `t = 0` as the visual anchor.


In [ ]:
detail_mask = (seconds_from_event >= -0.24) & (seconds_from_event <= 0.08)
detail_time = seconds_from_event[detail_mask]
detail_raw = strain[detail_mask]
detail_centered = np.nan_to_num(detail_raw - np.nanmean(detail_raw))
detail_frequency = np.fft.rfftfreq(detail_centered.size, d=x_spacing)
low_rolloff = np.clip((detail_frequency - 25) / 10, 0, 1)
high_rolloff = np.clip((400 - detail_frequency) / 50, 0, 1)
bandpass_response = np.minimum(low_rolloff, high_rolloff)
bandpassed_strain = (
    np.fft.irfft(
        np.fft.rfft(detail_centered * np.hanning(detail_centered.size)) * bandpass_response,
        n=detail_centered.size,
    )
    * 1e21
)

visible_domain = (-0.18, 0.05)
visible_mask = (detail_time >= visible_domain[0]) & (detail_time <= visible_domain[1])
visible_strain = bandpassed_strain[visible_mask]
detail_y_bound = max(4.0, np.ceil(np.nanpercentile(np.abs(visible_strain), 99.8) * 2) / 2)
late_mask = (detail_time >= -0.09) & (detail_time <= -0.05)
late_indices = np.flatnonzero(late_mask)
late_index = late_indices[np.argmax(bandpassed_strain[late_indices])]

bandpassed_chart = xy.line_chart(
    xy.line(
        detail_time,
        bandpassed_strain,
        name=f"{DETECTOR} · 35-350 Hz bandpass glow",
        color="#43f4ff",
        width=5.0,
        opacity=0.10,
    ),
    xy.line(
        detail_time,
        bandpassed_strain,
        name=f"{DETECTOR} · 35-350 Hz bandpass",
        color="#55e8ff",
        width=1.35,
        opacity=0.96,
    ),
    xy.hline(0, color="#2b7680", width=1, opacity=0.68),
    xy.vline(0, color="#ff4f91", width=2.25, opacity=0.98, style={"dash": "7,5"}),
    xy.text(
        0,
        detail_y_bound * 0.88,
        "GWOSC EVENT REFERENCE  /  t = 0",
        dx=-12,
        dy=-6,
        color="#ff4f91",
        anchor="end",
        style={"font_size": 12, "font_weight": 700, "letter_spacing": "0.06em"},
    ),
    xy.callout(
        float(detail_time[late_index]),
        float(bandpassed_strain[late_index]),
        "LATE INSPIRAL",
        dx=-86,
        dy=-46,
        color="#8ddfeb",
        width=1.25,
        anchor="end",
        style={
            "font_size": 12,
            "font_weight": 700,
            "label_color": "#d7f9fc",
            "letter_spacing": "0.06em",
        },
    ),
    xy.text(
        0.136,
        0.94,
        EVENT,
        dx=0,
        dy=0,
        color="#f1fdff",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 26,
            "font_weight": 700,
            "letter_spacing": "0.025em",
        },
    ),
    xy.text(
        0.136,
        0.895,
        f"{DETECTOR_SITES[DETECTOR]} {DETECTOR}  /  {1 / x_spacing:,.0f} HZ  /  35-350 HZ BANDPASS",
        dx=0,
        dy=0,
        color="#79a7af",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 10,
            "font_weight": 700,
            "letter_spacing": "0.09em",
        },
    ),
    xy.x_axis(
        label="TIME FROM GWOSC EVENT REFERENCE  /  seconds",
        domain=visible_domain,
        bounds=(float(detail_time[0]), float(detail_time[-1])),
        tick_values=[-0.15, -0.10, -0.05, 0, 0.05],
        tick_labels=["-0.15", "-0.10", "-0.05", "0", "+0.05"],
        style=SIGNAL_AXIS_STYLE,
    ),
    xy.y_axis(
        label="BANDPASSED STRAIN  /  \u00d7 10\u207b\u00b2\u00b9",
        label_offset=-18,
        domain=(-detail_y_bound, detail_y_bound),
        bounds=(float(np.nanmin(bandpassed_strain)), float(np.nanmax(bandpassed_strain))),
        tick_values=[-4, -2, 0, 2, 4],
        tick_labels=["-4", "-2", "0", "+2", "+4"],
        style=SIGNAL_AXIS_STYLE,
    ),
    xy.tooltip(
        title=f"{DETECTOR} 35-350 Hz BANDPASS",
        format={"x": "+.5f", "y": "+.3f"},
    ),
    xy.legend(show=False),
    xy.interaction_config(
        hover=True,
        crosshair=True,
        wheel_zoom=True,
        box_zoom=True,
        double_click_reset=True,
    ),
    SIGNAL_THEME,
    styles=SIGNAL_STYLES,
    style=SIGNAL_CARD,
    width=1150,
    height=620,
    padding=(96, 36, 80, 156),
)
detail_payload = bandpassed_chart.figure().build_payload()[0]
print("event-detail render tier:", detail_payload["traces"][1]["tier"])
bandpassed_chart

## Reconstructed waveform: the signal as the visual hero

GWOSC also publishes the numerical-relativity reference and reconstructed
Hanford strain used for its event figure. This final view peak-aligns the
reconstructed column, preserves its documented `strain × 10²¹` scale, and
labels the inspiral-to-ringdown story directly.


In [ ]:
reconstruction_seconds, _nr_strain, reconstructed_strain = np.loadtxt(
    reconstruction_path,
    comments="#",
    unpack=True,
)
peak_index = int(np.argmax(np.abs(reconstructed_strain)))
reconstruction_seconds = reconstruction_seconds - reconstruction_seconds[peak_index]
inspiral_index = int(np.argmin(np.abs(reconstruction_seconds + 0.028)))
ringdown_index = int(np.argmin(np.abs(reconstruction_seconds - 0.031)))
reconstruction_bounds = (
    float(np.nanmin(reconstructed_strain)),
    float(np.nanmax(reconstructed_strain)),
)

chart = xy.line_chart(
    xy.line(
        reconstruction_seconds,
        reconstructed_strain,
        name="H1 reconstructed strain · glow",
        color="#43f4ff",
        width=5.5,
        opacity=0.10,
    ),
    xy.line(
        reconstruction_seconds,
        reconstructed_strain,
        name="H1 reconstructed strain",
        color="#55e8ff",
        width=1.45,
        opacity=0.98,
    ),
    xy.hline(0, color="#2b7680", width=1, opacity=0.68),
    xy.vline(0, color="#ff4f91", width=2.25, opacity=0.98, style={"dash": "7,5"}),
    xy.text(
        0,
        1.40,
        "PEAK STRAIN  /  t = 0",
        dx=-12,
        dy=-6,
        color="#ff4f91",
        anchor="end",
        style={"font_size": 13, "font_weight": 700, "letter_spacing": "0.06em"},
    ),
    xy.callout(
        float(reconstruction_seconds[inspiral_index]),
        float(reconstructed_strain[inspiral_index]),
        "AMPLITUDE + FREQUENCY RISE",
        dx=-132,
        dy=-58,
        color="#8ddfeb",
        width=1.25,
        anchor="start",
        style={
            "font_size": 13,
            "font_weight": 700,
            "label_color": "#d7f9fc",
            "letter_spacing": "0.05em",
        },
    ),
    xy.callout(
        float(reconstruction_seconds[ringdown_index]),
        float(reconstructed_strain[ringdown_index]),
        "RINGDOWN",
        dx=-66,
        dy=-42,
        color="#ff6fa7",
        width=1.25,
        anchor="end",
        style={
            "font_size": 13,
            "font_weight": 700,
            "label_color": "#ffd3e4",
            "letter_spacing": "0.07em",
        },
    ),
    xy.text(
        0.137,
        0.94,
        EVENT,
        dx=0,
        dy=0,
        color="#f1fdff",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 26,
            "font_weight": 700,
            "letter_spacing": "0.025em",
        },
    ),
    xy.text(
        0.137,
        0.895,
        "HANFORD H1  /  OFFICIAL GWOSC TEMPLATE RECONSTRUCTION  /  PEAK-ALIGNED",
        dx=0,
        dy=0,
        color="#79a7af",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 10,
            "font_weight": 700,
            "letter_spacing": "0.085em",
        },
    ),
    xy.text(
        0.963,
        0.925,
        "14 SEP 2015  /  09:50:45 UTC",
        dx=0,
        dy=0,
        color="#ff6fa7",
        anchor="end",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 10,
            "font_weight": 700,
            "letter_spacing": "0.075em",
        },
    ),
    xy.x_axis(
        label="TIME FROM PEAK STRAIN  /  seconds",
        domain=(-0.15, 0.055),
        bounds=(float(reconstruction_seconds[0]), float(reconstruction_seconds[-1])),
        tick_values=[-0.15, -0.10, -0.05, 0, 0.05],
        tick_labels=["-0.15", "-0.10", "-0.05", "0", "+0.05"],
        style=SIGNAL_AXIS_STYLE,
    ),
    xy.y_axis(
        label="RECONSTRUCTED STRAIN  /  \u00d7 10\u207b\u00b2\u00b9",
        label_offset=-18,
        domain=(-1.55, 1.55),
        bounds=reconstruction_bounds,
        tick_values=[-1.0, -0.5, 0, 0.5, 1.0],
        tick_labels=["-1.0", "-0.5", "0", "+0.5", "+1.0"],
        style=SIGNAL_AXIS_STYLE,
    ),
    xy.tooltip(
        title="H1 GWOSC TEMPLATE RECONSTRUCTION",
        format={"x": "+.5f", "y": "+.3f"},
    ),
    xy.legend(show=False),
    xy.interaction_config(
        hover=True,
        crosshair=True,
        wheel_zoom=True,
        box_zoom=True,
        double_click_reset=True,
    ),
    SIGNAL_THEME,
    styles=SIGNAL_STYLES,
    style=SIGNAL_CARD,
    width=1200,
    height=700,
    padding=(102, 44, 90, 164),
)
chart